In [15]:
import numpy as np
import scipy.sparse as sp
from typing import Tuple

In [ ]:
from typing import List, Dict, Optional, Set
import time
from scipy.optimize import linprog

# ---------- Feasibility and Validation Utilities (Ax <= b) ---------- #

def is_feasible_ub(A, b) -> bool:
    """Return True iff Ax <= b is feasible. A can be dense or sparse."""
    m, n = A.shape
    c = np.zeros(n)
    res = linprog(c, A_ub=A, b_ub=b, method='highs')
    return bool(res.success)


def row_scale(A, b):
    """Scale each row i by s_i = 1 / max(1, ||A_i||_inf) to improve conditioning; return (A_s, b_s, s)."""
    A_is_sparse = sp.issparse(A)
    m, n = A.shape
    if A_is_sparse:
        A_csr = A.tocsr().astype(float)
        s = np.zeros(m, dtype=float)
        for i in range(m):
            row = A_csr.getrow(i)
            mx = np.max(np.abs(row.data)) if row.nnz > 0 else 0.0
            s[i] = 1.0 / max(1.0, mx)
        D = sp.diags(s)
        A_s = D @ A_csr
    else:
        mx = np.max(np.abs(A), axis=1)
        s = 1.0 / np.maximum(1.0, mx)
        A_s = (s[:, None] * A)
    b_s = s * b
    return (A_s, b_s, s)


def validate_conflict_sets(A, b, V_sets: List[List[int]], check_minimal: bool = True) -> Dict:
    """Validate that each V_i induces infeasibility and union removal yields feasibility.
    Returns a dict summary with per-set results and overall feasibility after removal.
    """
    A_is_sparse = sp.issparse(A)
    m = A.shape[0]

    per_set = []
    union_rows: Set[int] = set()
    for idx, V in enumerate(V_sets):
        V_sorted = sorted(set(int(i) for i in V))
        union_rows.update(V_sorted)
        A_sub = A[V_sorted, :] if A_is_sparse else A[V_sorted, :]
        b_sub = b[V_sorted]
        infeasible = not is_feasible_ub(A_sub, b_sub)
        minimal_ok = True
        if check_minimal and infeasible and len(V_sorted) >= 2:
            for j in V_sorted:
                rows_minus = [r for r in V_sorted if r != j]
                A_minus = A[rows_minus, :] if A_is_sparse else A[rows_minus, :]
                b_minus = b[rows_minus]
                # Removing any single constraint from an IIS must make it feasible
                if not is_feasible_ub(A_minus, b_minus):
                    minimal_ok = False
                    break
        per_set.append({
            'set_index': idx,
            'size': len(V_sorted),
            'infeasible': infeasible,
            'minimal': minimal_ok,
        })

    # Check feasibility after removing union of all V_i rows
    keep = sorted(set(range(m)) - union_rows)
    A_keep = A[keep, :] if A_is_sparse else A[keep, :]
    b_keep = b[keep]
    feasible_after_removal = is_feasible_ub(A_keep, b_keep)

    return {
        'per_set': per_set,
        'feasible_after_removal': feasible_after_removal,
        'removed_rows_count': len(union_rows),
        'remaining_rows_count': len(keep),
    }

# ---------- Farkas Certificate (Dual) and IIS Extraction ---------- #

def find_farkas_certificate(A, b, sum_normalize: bool = True, neg_tol: float = 1e-8) -> Optional[np.ndarray]:
    """Find y >= 0 with A^T y = 0 and b^T y < 0 (Farkas certificate) if it exists.
    Tries multiple robust formulations (sum(y)=1 or sum(y)<=1; sparse/dense) to mitigate numerical issues.
    Returns y or None if no certificate was found (i.e., likely Ax<=b is feasible).
    """
    m, n = A.shape
    c = b.astype(float)

    # Build helpers to attempt variants
    def attempt(A_eq=None, b_eq=None, A_ub=None, b_ub=None):
        bounds = [(0, None)] * m
        res = linprog(c, A_eq=A_eq, b_eq=b_eq, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
        if res.success and res.fun < -neg_tol:
            return res.x
        return None

    # Variant 1: sum(y)=1 equality (sparse or dense as provided)
    if sum_normalize:
        if sp.issparse(A):
            A_eq = sp.vstack([A.T, sp.csr_matrix(np.ones((1, m)))])
        else:
            A_eq = np.vstack([A.T, np.ones((1, m))])
        b_eq = np.concatenate([np.zeros(n), np.array([1.0])])
        y = attempt(A_eq=A_eq, b_eq=b_eq)
        if y is not None:
            return y
    else:
        A_eq = A.T
        b_eq = np.zeros(n)
        y = attempt(A_eq=A_eq, b_eq=b_eq)
        if y is not None:
            return y

    # Variant 2: sum(y) <= 1 inequality normalization
    A_eq2 = A.T
    b_eq2 = np.zeros(n)
    if sp.issparse(A):
        A_ub = sp.csr_matrix(np.ones((1, m)))
    else:
        A_ub = np.ones((1, m))
    b_ub = np.array([1.0])
    y = attempt(A_eq=A_eq2, b_eq=b_eq2, A_ub=A_ub, b_ub=b_ub)
    if y is not None:
        return y

    # Variant 3: Dense fallback (some HiGHS paths are more stable on dense)
    A_dense = A.toarray() if sp.issparse(A) else A
    A_eq_dense = np.vstack([A_dense.T, np.ones((1, m))])
    b_eq_dense = np.concatenate([np.zeros(n), np.array([1.0])])
    y = attempt(A_eq=A_eq_dense, b_eq=b_eq_dense)
    if y is not None:
        return y

    return None


def reduce_to_iis_by_deletion(A, b, rows: List[int]) -> List[int]:
    """Given an infeasible set of row indices, reduce it to an IIS by deletion tests."""
    A_is_sparse = sp.issparse(A)
    S = list(sorted(set(int(i) for i in rows)))
    changed = True
    while changed and len(S) >= 2:
        changed = False
        for i in list(S):
            S_minus = [r for r in S if r != i]
            A_sub = A[S_minus, :] if A_is_sparse else A[S_minus, :]
            b_sub = b[S_minus]
            # If still infeasible without i, drop i as redundant
            if not is_feasible_ub(A_sub, b_sub):
                S.remove(i)
                changed = True
    return S


def find_iis_sets_via_farkas(A, b, support_tol: float = 1e-8, check_minimal: bool = True):
    """Iteratively find disjoint conflict sets using Farkas certificates and deletion reduction.
    Returns list of V_i (lists of row indices in original coordinates)."""
    # Row-scale for robustness
    A_s, b_s, s = row_scale(A, b)

    A_is_sparse = sp.issparse(A_s)
    m = A_s.shape[0]
    # Track mapping from current row indices to original
    current_to_orig = list(range(m))
    A_cur, b_cur = A_s, b_s
    V_sets: List[List[int]] = []

    while True:
        # Stop if remaining system is feasible
        if is_feasible_ub(A_cur, b_cur):
            break
        y = find_farkas_certificate(A_cur, b_cur)
        if y is None:
            break  # No certificate found, likely feasible or numerical issue
        # Normalize y for stable support extraction
        y = np.maximum(y, 0.0)
        sy = y / (np.max(y) + 1e-12)
        support = [i for i, v in enumerate(sy) if v >= max(support_tol, 0.1)]  # keep strong signals only
        if len(support) < 2:
            # fallback: take top-3 indices by y if available
            order = np.argsort(-sy)
            support = list(order[:min(3, len(order))])
        # Reduce to an IIS
        iis_cur = reduce_to_iis_by_deletion(A_cur, b_cur, support)
        # Map back to original row indices (A_s rows map 1:1 to original here)
        iis_orig = [current_to_orig[i] for i in iis_cur]
        V_sets.append(sorted(iis_orig))
        # Remove these rows from the current system and continue
        keep_mask = np.ones(len(current_to_orig), dtype=bool)
        keep_mask[iis_cur] = False
        keep_rows = [idx for idx, flag in enumerate(keep_mask) if flag]
        if len(keep_rows) == 0:
            break
        A_cur = A_cur[keep_rows, :] if A_is_sparse else A_cur[keep_rows, :]
        b_cur = b_cur[keep_rows]
        current_to_orig = [current_to_orig[k] for k in keep_rows]

    # Optionally check minimality of the found sets on the ORIGINAL scale
    if check_minimal and V_sets:
        V_sets = [reduce_to_iis_by_deletion(A, b, V) for V in V_sets]
    return V_sets

In [33]:
# ---------- End-to-End Test Harness ---------- #

# Helper to generate a system with known conflict sets in Ax <= b form

def make_sparse_infeasible_with_truth(
    num_equations: int = 400,
    num_variables: int = 200,
    num_conflicts: int = 3,
    conflict_size: int = 6,
    seed: int = 0,
):
    rng = np.random.default_rng(seed)
    # Sparse builder for incremental edits
    A = sp.lil_matrix((num_equations, num_variables), dtype=float)
    b = np.zeros(num_equations, dtype=float)

    total_conf = num_conflicts * conflict_size
    assert total_conf > 0 and total_conf <= num_equations

    # Choose conflict rows as the last contiguous block to avoid overlap with filler
    conflict_rows = list(range(num_equations - total_conf, num_equations))
    filler_rows = list(range(0, num_equations - total_conf))

    # Build a guaranteed-feasible base using a random x_true and positive slack
    x_true = rng.uniform(0.0, 5.0, size=num_variables)
    for i in filler_rows:
        nz = int(rng.integers(2, min(6, num_variables) + 1))
        cols = rng.choice(num_variables, size=nz, replace=False)
        vals = rng.uniform(-2.0, 2.0, size=nz)
        A[i, cols] = vals
    b_fill = (A[filler_rows, :] @ x_true)
    if sp.issparse(b_fill):
        b_fill = np.asarray(b_fill).reshape(-1)
    # Add positive slack to ensure Ax_true <= b on filler rows
    b[filler_rows] = b_fill + rng.uniform(1.0, 3.0, size=len(filler_rows))

    # Embed disjoint conflicts: (k-1) lower bounds and one tight sum upper bound
    V_truth: List[List[int]] = []
    for k in range(num_conflicts):
        rows = conflict_rows[k * conflict_size : (k + 1) * conflict_size]
        # Important: clear any pre-existing coefficients from these rows
        for r in rows:
            A[r, :] = 0.0
        vars_for_conflict = rng.choice(num_variables, size=conflict_size - 1, replace=False)
        constants = rng.integers(5, 15, size=conflict_size - 1)
        # -x_j <= -c_j (i.e., x_j >= c_j)
        for j in range(conflict_size - 1):
            A[rows[j], vars_for_conflict[j]] = -1.0
            b[rows[j]] = -float(constants[j])
        # sum x_j <= sum(c_j) - 1 (infeasible with the above lower bounds)
        srow = rows[-1]
        A[srow, vars_for_conflict] = 1.0
        b[srow] = float(np.sum(constants) - 1)
        V_truth.append(sorted(rows))

    return A.tocsc(), b, V_truth


# Simple exact-match check for sets (as sets of frozensets)

def exact_match_sets(found: List[List[int]], truth: List[List[int]]) -> bool:
    norm = lambda S: frozenset(sorted(int(i) for i in S))
    return set(map(norm, found)) == set(map(norm, truth))


# Build a test, run algorithm, validate, and print concise report

def run_experiment():
    A, b, V_truth = make_sparse_infeasible_with_truth(
        num_equations=400, num_variables=200, num_conflicts=1, conflict_size=70, seed=42
    )

    t0 = time.perf_counter()
    V_found = find_iis_sets_via_farkas(A, b)
    t_algo = time.perf_counter() - t0

    # Validate proposed sets and overall feasibility after removal
    t1 = time.perf_counter()
    val = validate_conflict_sets(A, b, V_found)
    t_val = time.perf_counter() - t1

    correct = exact_match_sets(V_found, V_truth)

    print("=== Experiment Report ===")
    print(f"Algorithm time (s): {t_algo:.4f}")
    print(f"Validation time (s): {t_val:.4f}")
    print(f"Sets found: {len(V_found)} | Truth: {len(V_truth)}")
    print(f"Exact match with truth: {correct}")
    print(f"Feasible after removing union: {val['feasible_after_removal']}")
    for i, info in enumerate(val['per_set']):
        print(f"  - V[{i}]: size={info['size']}, infeasible={info['infeasible']}, minimal={info['minimal']}")


# Run the experiment
run_experiment()

=== Experiment Report ===
Algorithm time (s): 0.1407
Validation time (s): 0.0233
Sets found: 2 | Truth: 1
Exact match with truth: False
Feasible after removing union: False
  - V[0]: size=3, infeasible=True, minimal=True
  - V[1]: size=2, infeasible=True, minimal=True


In [ ]:
from scipy.sparse.linalg import spsolve

def make_spd_banded(n: int):
    # 5-diagonal, strictly diagonally dominant SPD matrix
    main = 4.0 * np.ones(n)
    off1 = -1.0 * np.ones(n - 1)
    off2 = -0.25 * np.ones(n - 2)
    return sp.diags([off2, off1, main, off1, off2], offsets=[-2, -1, 0, 1, 2], format='csr')

# Benchmark spsolve (sparse) vs np.linalg.solve (dense)
sizes = [500, 1000]  # increase carefully (dense O(n^3) can be slow)
repeats = 3
rng = np.random.default_rng(123)

for n in sizes:
    A_mat = make_spd_banded(n)
    x_true = rng.standard_normal(n)
    b_vec = A_mat @ x_true

    # Sparse solve
    t_best_sparse = float('inf')
    x_sparse = None
    for _ in range(repeats):
        t0 = time.perf_counter()
        x_candidate = spsolve(A_mat, b_vec)
        t = time.perf_counter() - t0
        if t < t_best_sparse:
            t_best_sparse = t
            x_sparse = x_candidate

    # Dense solve (convert once)
    t0 = time.perf_counter()
    A_dense = A_mat.toarray()
    t_dense_convert = time.perf_counter() - t0

    t_best_dense = float('inf')
    x_dense = None
    for _ in range(repeats):
        t0 = time.perf_counter()
        x_candidate = np.linalg.solve(A_dense, b_vec)
        t = time.perf_counter() - t0
        if t < t_best_dense:
            t_best_dense = t
            x_dense = x_candidate

    # Accuracy checks
    rel_res_sparse = np.linalg.norm(A_mat @ x_sparse - b_vec) / np.linalg.norm(b_vec)
    rel_res_dense = np.linalg.norm(A_dense @ x_dense - b_vec) / np.linalg.norm(b_vec)

    nnz = A_mat.nnz
    density = nnz / (n * n)
    print(f"n={n} | nnz={nnz} ({density:.4%} dense)")
    print(f"  spsolve:      {t_best_sparse:.4f}s | rel_res={rel_res_sparse:.2e}")
    print(f"  np.linalg:    {t_best_dense:.4f}s (+{t_dense_convert:.4f}s to densify) | rel_res={rel_res_dense:.2e}")
    if t_best_sparse > 0:
        print(f"  speedup (dense/sparse) ~ {t_best_dense / t_best_sparse:.2f}x")
    print()